# RASCAR V17: воспроизведение

Основной CSV: `submission_v17_main.csv`. OOF 0.981816; новый LB неизвестен.

В архиве соседние папки `rascar_v16` и `rascar_v17`: сохраните их расположение. Этот ноутбук воспроизводит сборку из сохранённых CV-предсказаний и признаков. Он не переобучает весь исторический baseline. Исходники новых GPU-экспериментов, начальные и новые веса сохранены.


In [ ]:
from pathlib import Path
import sys, subprocess, json, hashlib
candidates = [Path.cwd(), Path.cwd()/'rascar_v17']
A = next((p for p in candidates if (p/'release_manifest.json').is_file() and (p/'common17.py').is_file()), None)
assert A is not None, 'Open this notebook from the extracted rascar_v17 folder.'
A = A.resolve()
manifest = json.loads((A/'release_manifest.json').read_text())
print(A, manifest['main'])


## CPU: точное восстановление CSV

Зависимости зафиксированы в `requirements-replay.txt`; при необходимости установите их в отдельное окружение.


In [ ]:
# Optional setup:
# subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(A/'requirements-replay.txt')], check=True)
subprocess.run([sys.executable, str(A/'replay.py')], cwd=A, check=True)


In [ ]:
import pandas as pd
display(pd.read_csv(A/'release_metric_comparison.csv'))
display(pd.read_csv(A/'audit_closure_counts.csv'))
for name, result in manifest['release_experiments'].items():
    p = A/'results/submissions'/f'submission_{name}.csv'
    assert hashlib.sha256(p.read_bytes()).hexdigest() == result['sha256']
    print(name, result['sha256'])


## Разделение дней и аудит

Тренировка на 25 октября -> предсказание 26; тренировка на 26 -> предсказание 25. Исходные 198 официально размеченных запросов сохранены без изменения меток. Порог IoU 0.6; высокий score означает ошибку запроса.

Тестовая визуальная оценка субъективна, использовалась для разработки, не является независимой валидацией и не использовалась как целевая разметка обучения. Диагностический процент прохождения не означает доказанную долю исправлений на скрытой разметке.


## Повтор нового GPU-этапа на Kaggle (опционально)

Понадобится прикрепить распакованный архив как входные данные и указать `BUNDLE`. Этот этап запускает три обучения YOLO26s: два дневных и одно на всех 84 размеченных кадрах. По 20 дополнительных эпох, checkpoint last, без подбора эпох по проверочному дню. Затем одинаковый TTA для старых и новых весов. Займёт GPU-время.

`gpu_payload.zip` содержит точные начальные веса и таблицы; фотографии уже находятся в соседней папке V16. 280 лишних кадров из общего списка не нужны для обучения/инференса и ниже отфильтрованы.


In [ ]:
# Run only in Kaggle when intentionally repeating GPU training:
# BUNDLE = Path('/kaggle/input/YOUR_EXTRACTED_BUNDLE')
# import zipfile, shutil
# work = Path('/kaggle/working/rascar_v17')
# work.mkdir(parents=True, exist_ok=True)
# with zipfile.ZipFile(BUNDLE/'rascar_v17/gpu_payload.zip') as z:
#     z.extractall(work)
# shutil.copy2(BUNDLE/'rascar_v17/gpu_launcher.py', work/'gpu_launcher.py')
# shutil.copy2(BUNDLE/'rascar_v17/gpu_worker.py', work/'gpu_worker.py')
# (work/'images').mkdir(exist_ok=True)
# for folder in ['images', 'test_images']:
#     for image in (BUNDLE/'rascar_v16'/folder).glob('*.jpg'):
#         shutil.copy2(image, work/'images'/image.name)
# frames = pd.read_csv(work/'frames.csv')
# needed = set(pd.read_csv(work/'manual_boxes.csv').frame_id) | set(pd.read_csv(work/'test_queries.csv').frame_id)
# frames[frames.frame_id.isin(needed)].to_csv(work/'frames.csv', index=False)
# subprocess.run([sys.executable, str(work/'gpu_launcher.py')], check=True)


## Все эксперименты

`all_experiment_metrics.csv` содержит удачные и неудачные варианты. Полная галерея: `galleries/release_balanced/index.html`. Все исходные версии V15/V16 сохранены. Новая оценка LB появится только после загрузки CSV на соревнование.
